In [1]:
# Import necessary libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number, rank, dense_rank, sum, avg
from pyspark.sql.window import Window

# Initialize Spark session
spark = SparkSession.builder \
    .appName("WindowFunctionsDemo") \
    .getOrCreate()

In [2]:
# Create DataFrame
data = [
    (1, "John", "Sales", 3000),
    (2, "Emily", "Sales", 4000),
    (3, "Michael", "Sales", 4000),  # Tie in Sales
    (4, "Sarah", "HR", 4500),
    (5, "David", "HR", 4000),
    (6, "Alice", "HR", 4000),       # Tie in HR
    (7, "Bob", "IT", 4800),
    (8, "Charlie", "IT", 5200),
    (9, "Diana", "IT", 5100)        
]

columns = ["id", "name", "department", "salary"]

df = spark.createDataFrame(data, columns)

print("Original DataFrame:")
df.show()

Original DataFrame:
+---+-------+----------+------+
| id|   name|department|salary|
+---+-------+----------+------+
|  1|   John|     Sales|  3000|
|  2|  Emily|     Sales|  4000|
|  3|Michael|     Sales|  4000|
|  4|  Sarah|        HR|  4500|
|  5|  David|        HR|  4000|
|  6|  Alice|        HR|  4000|
|  7|    Bob|        IT|  4800|
|  8|Charlie|        IT|  5200|
|  9|  Diana|        IT|  5100|
+---+-------+----------+------+



In [3]:
# Define a window specification
window_spec = Window.partitionBy("department").orderBy(col("salary").desc())

In [4]:
# 1. Row Number
row_number_df = df.withColumn("row_number", row_number().over(window_spec))
print("Row Number:")
row_number_df.show()

Row Number:
+---+-------+----------+------+----------+
| id|   name|department|salary|row_number|
+---+-------+----------+------+----------+
|  4|  Sarah|        HR|  4500|         1|
|  5|  David|        HR|  4000|         2|
|  6|  Alice|        HR|  4000|         3|
|  8|Charlie|        IT|  5200|         1|
|  9|  Diana|        IT|  5100|         2|
|  7|    Bob|        IT|  4800|         3|
|  2|  Emily|     Sales|  4000|         1|
|  3|Michael|     Sales|  4000|         2|
|  1|   John|     Sales|  3000|         3|
+---+-------+----------+------+----------+



In [5]:
# 2. Rank
rank_df = df.withColumn("rank", rank().over(window_spec))
print("Rank:")
rank_df.show()

Rank:
+---+-------+----------+------+----+
| id|   name|department|salary|rank|
+---+-------+----------+------+----+
|  4|  Sarah|        HR|  4500|   1|
|  5|  David|        HR|  4000|   2|
|  6|  Alice|        HR|  4000|   2|
|  8|Charlie|        IT|  5200|   1|
|  9|  Diana|        IT|  5100|   2|
|  7|    Bob|        IT|  4800|   3|
|  2|  Emily|     Sales|  4000|   1|
|  3|Michael|     Sales|  4000|   1|
|  1|   John|     Sales|  3000|   3|
+---+-------+----------+------+----+



In [6]:
# 3. Dense Rank
dense_rank_df = df.withColumn("dense_rank", dense_rank().over(window_spec))
print("Dense Rank:")
dense_rank_df.show()

Dense Rank:
+---+-------+----------+------+----------+
| id|   name|department|salary|dense_rank|
+---+-------+----------+------+----------+
|  4|  Sarah|        HR|  4500|         1|
|  5|  David|        HR|  4000|         2|
|  6|  Alice|        HR|  4000|         2|
|  8|Charlie|        IT|  5200|         1|
|  9|  Diana|        IT|  5100|         2|
|  7|    Bob|        IT|  4800|         3|
|  2|  Emily|     Sales|  4000|         1|
|  3|Michael|     Sales|  4000|         1|
|  1|   John|     Sales|  3000|         2|
+---+-------+----------+------+----------+

